# 🚀 Pipeline Tự Động Huấn Luyện 5 Cấu Hình Ablation Cho Scene `teapot` Trên Google Colab

Notebook này thực hiện trọn vẹn **5 thực nghiệm (5 Runs)** trên scene `teapot`, tích hợp cơ chế **Thông Minh Tự Động Skip (Bỏ qua các run đã chạy thành công)** để bạn có thể an tâm bấm `Run All` mà không bao giờ bị train lại tốn thời gian:

### 🎯 Danh Sách 5 Cấu Hình Thực Nghiệm:
1. **RUN 1 (Full Method / Baseline)**: Bật đủ 3 cờ `--use_ref_score`, `--use_adaptive_prior`, `--use_reflection_view_sampling` $\rightarrow$ Output `teapot`
2. **RUN 2 (Disable Multiview)**: Bật cờ `--disable_multiview_contribution` $\rightarrow$ Output `teapot_disable_multiview`
3. **RUN 3 (Only Ref Score)**: Chỉ bật `--use_ref_score` (tắt adaptive và view sampling) $\rightarrow$ Output `teapot_only_ref_score`
4. **RUN 4 (Ref Score + Adaptive Prior)**: Bật `--use_ref_score` và `--use_adaptive_prior` (tắt view sampling) $\rightarrow$ Output `teapot_ref_score_adaptive`
5. **RUN 5 (No Reflection Prior / Vanilla)**: Tắt toàn bộ 3 cờ `use_*` $\rightarrow$ Output `teapot_no_reflection`

---

## 🛠️ Bước 1: Xác Thực Google Account (Auth) & Mount Google Drive

> **Lưu ý**: Chạy cell dưới đây sẽ xác thực tài khoản Google và kết nối với `/content/drive/MyDrive`.

In [ ]:
# ── 1. Xác Thực Google Colab & Mount Google Drive ─────────────────────────────
import os
import sys
import shutil
import time
import io
import json
from pathlib import Path

# Xác thực tài khoản Google để truy cập trực tiếp các thư mục 'Shared with me'
try:
    from google.colab import auth
    print("🔐 Đang xác thực tài khoản Google Colab...")
    auth.authenticate_user()
    print("✅ Xác thực Google Account thành công!")
except Exception as e:
    print(f"ℹ️ Chạy môi trường ngoài Colab hoặc đã xác thực: {e}")

# Mount Google Drive
try:
    from google.colab import drive
    print("🔄 Đang mount Google Drive vào /content/drive...")
    drive.mount('/content/drive/')
    print("✅ Google Drive đã được mount thành công tại /content/drive/MyDrive!")
except Exception as e:
    print(f"ℹ️ Mount drive note: {e}")

# Cài đặt các thư viện cần thiết
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib plyfile tqdm websockets openpyxl pandas ninja

# Kiểm tra GPU
import torch
print("-" * 65)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU Hoạt động : {gpu_name}")
    print(f"✅ CUDA Version  : {torch.version.cuda}")
    print(f"✅ PyTorch Ver   : {torch.__version__}")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Vào Runtime -> Change runtime type -> Chọn T4 GPU hoặc A100 GPU.")
print("-" * 65)

# ── 2. Khai Báo Các ID & Đường Dẫn Google Drive ───────────────────────────────
MYDRIVE_ROOT = "/content/drive/MyDrive"

# 1. Thư mục chứa Source code (Hình 1)
GDRIVE_SOURCE_FOLDER_ID = "10pqXcWVPrMA47NhJJ_LZiLKjWSaQ-ZkQ"
SOURCE_DEST_DIR = os.path.join(MYDRIVE_ROOT, "Thesis", "source", "20082026")
REPO_DIR = os.path.join(SOURCE_DEST_DIR, "spec-fastgs")

# 2. Thư mục chứa Dataset (Hình 2)
GDRIVE_DATASET_FOLDER_ID = "1hH7qMSbTyR392PYgsqeMhAnaAxwxzemc"
GDRIVE_DATASET_DIR = os.path.join(MYDRIVE_ROOT, "Anisotropic-Synthetic-Dataset")
SCENE_NAME = "teapot"
DATASET_TEAPOT_DIR = os.path.join(GDRIVE_DATASET_DIR, SCENE_NAME)

# 3. Thư mục Upload Kết Quả (Hình 3)
GDRIVE_RESULTS_FOLDER_ID = "1FZdVaZmY9VsFMGXWc5107z4pGz-cTyCh"
GDRIVE_RESULTS_DIR = os.path.join(MYDRIVE_ROOT, "Thesis", "Results", "20082026")

print("📁 Các đường dẫn chính:")
print(f"   - REPO_DIR          : {REPO_DIR}")
print(f"   - DATASET_TEAPOT_DIR: {DATASET_TEAPOT_DIR}")
print(f"   - GDRIVE_RESULTS_DIR: {GDRIVE_RESULTS_DIR}")

# ── 3. Khởi Tạo Google Drive API Client ────────────────────────────────────────
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import google.auth

credentials, project = google.auth.default()
drive_service = build('drive', 'v3', credentials=credentials)

def list_drive_folder_items(folder_id):
    """Liệt kê toàn bộ file/folder con bên trong một folder ID Google Drive."""
    items = []
    page_token = None
    while True:
        query = f"'{folder_id}' in parents and trashed = false"
        response = drive_service.files().list(
            q=query,
            spaces='drive',
            fields='nextPageToken, files(id, name, mimeType, size)',
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()
        items.extend(response.get('files', []))
        page_token = response.get('nextPageToken', None)
        if page_token is None:
            break
    return items

def download_drive_file_by_id(file_id, dest_filepath):
    """Tải file nhị phân từ Google Drive theo File ID."""
    os.makedirs(os.path.dirname(dest_filepath), exist_ok=True)
    request = drive_service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with io.FileIO(dest_filepath, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=10*1024*1024)
        done = False
        while not done:
            status, done = downloader.next_chunk()

def download_drive_folder_recursive(folder_id, local_dir):
    """Tải đệ quy toàn bộ cây thư mục từ Drive về local directory."""
    os.makedirs(local_dir, exist_ok=True)
    items = list_drive_folder_items(folder_id)
    for item in items:
        name = item['name']
        item_id = item['id']
        mime = item['mimeType']
        if mime == 'application/vnd.google-apps.folder':
            download_drive_folder_recursive(item_id, os.path.join(local_dir, name))
        else:
            local_path = os.path.join(local_dir, name)
            if not os.path.exists(local_path):
                download_drive_file_by_id(item_id, local_path)

def create_or_get_drive_folder(parent_id, folder_name):
    """Tìm hoặc tạo thư mục con trên Drive theo parent_id."""
    items = list_drive_folder_items(parent_id)
    for item in items:
        if item['name'] == folder_name and item['mimeType'] == 'application/vnd.google-apps.folder':
            return item['id']
    file_metadata = {
        'name': folder_name,
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [parent_id]
    }
    folder = drive_service.files().create(body=file_metadata, fields='id', supportsAllDrives=True).execute()
    return folder.get('id')

def upload_local_folder_to_drive(local_dir, parent_folder_id, folder_name, exclude_names=None):
    """Upload một thư mục local lên Google Drive qua API (hỗ trợ loại trừ thư mục như 'test')."""
    if exclude_names is None:
        exclude_names = []
    exclude_lower = [x.lower() for x in exclude_names]

    print(f"📤 Đang tải '{folder_name}' lên Drive (Parent ID: {parent_folder_id})...")
    target_folder_id = create_or_get_drive_folder(parent_folder_id, folder_name)

    folder_id_map = {local_dir: target_folder_id}

    for root, dirs, files in os.walk(local_dir):
        dirs[:] = [d for d in dirs if d.lower() not in exclude_lower]
        current_drive_id = folder_id_map[root]
        existing_drive_items = {item['name']: item for item in list_drive_folder_items(current_drive_id)}

        for d in dirs:
            sub_local_path = os.path.join(root, d)
            if d in existing_drive_items and existing_drive_items[d]['mimeType'] == 'application/vnd.google-apps.folder':
                sub_drive_id = existing_drive_items[d]['id']
            else:
                sub_drive_id = create_or_get_drive_folder(current_drive_id, d)
            folder_id_map[sub_local_path] = sub_drive_id

        for f in files:
            file_path = os.path.join(root, f)
            if f in existing_drive_items:
                continue
            try:
                media = MediaFileUpload(file_path, resumable=True)
                drive_service.files().create(
                    body={'name': f, 'parents': [current_drive_id]},
                    media_body=media,
                    fields='id',
                    supportsAllDrives=True
                ).execute()
            except Exception as e:
                print(f"   ⚠️ Lỗi tải file {f}: {e}")
    print(f"✅ Đã upload thành công thư mục '{folder_name}' lên Google Drive!")

🔐 Đang xác thực tài khoản Google Colab...
✅ Xác thực Google Account thành công!
🔄 Đang mount Google Drive vào /content/drive...
Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
✅ Google Drive đã được mount thành công tại /content/drive/MyDrive!
-----------------------------------------------------------------
✅ GPU Hoạt động : Tesla T4
✅ CUDA Version  : 12.8
✅ PyTorch Ver   : 2.11.0+cu128
-----------------------------------------------------------------
📁 Các đường dẫn chính:
   - REPO_DIR          : /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs
   - DATASET_TEAPOT_DIR: /content/drive/MyDrive/Anisotropic-Synthetic-Dataset/teapot
   - GDRIVE_RESULTS_DIR: /content/drive/MyDrive/Thesis/Results/20082026


## 📦 Bước 2: Chuẩn Bị Mã Nguồn `spec-fastgs` Từ Google Drive [Hình 1]

> Tự động kiểm tra và chỉ tải/giải nén đúng **1 lần duy nhất** vào `/content/drive/MyDrive/Thesis/source/20082026/spec-fastgs`.

In [ ]:
# ── Kiểm tra & Tải Mã Nguồn spec-fastgs (Chỉ làm 1 lần) ───────────────────────
import zipfile

os.makedirs(SOURCE_DEST_DIR, exist_ok=True)
is_source_ready = os.path.isdir(REPO_DIR) and os.path.exists(os.path.join(REPO_DIR, "train.py"))

if is_source_ready:
    print(f"✅ Mã nguồn spec-fastgs đã sẵn sàng tại: {REPO_DIR}")
    print("⏩ Bỏ qua bước tải và giải nén file .zip!")
else:
    print(f"🔍 Đang chuẩn bị mã nguồn trong MyDrive: {SOURCE_DEST_DIR}...")
    items = list_drive_folder_items(GDRIVE_SOURCE_FOLDER_ID)
    zip_item = next((i for i in items if i['name'].endswith('.zip')), None)

    if zip_item:
        zip_name = zip_item['name']
        local_zip_path = os.path.join(SOURCE_DEST_DIR, zip_name)
        if not os.path.exists(local_zip_path):
            print(f"📥 Đang tải {zip_name} (ID: {zip_item['id']})...")
            download_drive_file_by_id(zip_item['id'], local_zip_path)
            print(f"✅ Đã tải xong: {local_zip_path}")

        print(f"📂 Đang giải nén {zip_name}...")
        with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
            zip_ref.extractall(SOURCE_DEST_DIR)
        print(f"✅ Giải nén mã nguồn thành công tại: {SOURCE_DEST_DIR}")
    else:
        print(f"📥 Tải toàn bộ thư mục từ Folder ID: {GDRIVE_SOURCE_FOLDER_ID}...")
        download_drive_folder_recursive(GDRIVE_SOURCE_FOLDER_ID, SOURCE_DEST_DIR)
        print(f"✅ Đã đồng bộ mã nguồn vào: {SOURCE_DEST_DIR}")

# Kiểm tra cấu trúc repo
if not os.path.exists(REPO_DIR):
    for item in os.listdir(SOURCE_DEST_DIR):
        candidate = os.path.join(SOURCE_DEST_DIR, item)
        if os.path.isdir(candidate) and os.path.exists(os.path.join(candidate, "train.py")):
            REPO_DIR = candidate
            break

print(f"\n🚀 REPO ROOT: {REPO_DIR}")
!ls -la "{REPO_DIR}"

✅ Mã nguồn spec-fastgs đã sẵn sàng tại: /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs
⏩ Bỏ qua bước tải và giải nén file .zip!

🚀 REPO ROOT: /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs
total 3178
drwx------ 3 root root    4096 Aug 20 05:02 arguments
drwx------ 2 root root    4096 Aug 20 05:02 assets
-rw------- 1 root root  283920 Aug 20 07:31 cameras.json
-rw------- 1 root root    5349 Aug 20 05:02 convert.py
-rw------- 1 root root     558 Aug 20 05:02 count_spec.py
drwx------ 5 root root    4096 Aug 20 07:24 datasets
-rw------- 1 root root   12020 Aug 20 05:02 diff_gaussian_model.txt
-rw------- 1 root root   65074 Aug 20 05:02 draft.md
-rw------- 1 root root     349 Aug 20 05:02 environment.yml
-rwx------ 1 root root    2128 Aug 20 15:25 exec_run1_teapot.sh
-rw------- 1 root root    1972 Aug 20 11:39 exec_run2_teapot_disable_multiview.sh
-rwx------ 1 root root    2157 Aug 20 15:28 exec_run3_teapot_teapot_only_ref_score.sh
drwx------ 2 root root    4096 Aug 20

## ⚡ Bước 3: Cài Đặt Submodules C++/CUDA (Kèm Cơ Chế Cache Wheel Không Phải Build Lại)

> Tận dụng các file `.whl` đã build sẵn lưu trong `/content/drive/MyDrive/Thesis/wheels_cache` để cài đặt chỉ trong **2 giây**!

In [ ]:
# ── 1. Chuẩn Bị & Build Wheel Caching Cho Submodules ───────────────────────────
import os
import subprocess
import glob

WHEELS_CACHE_DIR = os.path.join(MYDRIVE_ROOT, "Thesis", "wheels_cache")
os.makedirs(WHEELS_CACHE_DIR, exist_ok=True)
SUBMODULES_DIR = os.path.join(REPO_DIR, "submodules")

submodule_names = [
    "diff-gaussian-rasterization_fastgs",
    "simple-knn",
    "fused-ssim"
]

print(f"📁 Thư mục Cache Wheels : {WHEELS_CACHE_DIR}")
print(f"🔨 Thư mục Submodules gốc: {SUBMODULES_DIR}")

# ── 2. Kiểm tra Wheel đã có sẵn hay cần build ─────────────────────────────────
for sub_name in submodule_names:
    sub_path = os.path.join(SUBMODULES_DIR, sub_name)
    clean_name = sub_name.replace("-", "_")
    cached_whl = glob.glob(os.path.join(WHEELS_CACHE_DIR, f"{clean_name}*.whl"))

    if cached_whl:
        print(f"⚡ Tìm thấy pre-built wheel cho '{sub_name}': {os.path.basename(cached_whl[0])}")
        print(f"   🚀 Cài đặt wheel siêu tốc...")
        !pip install --no-deps --force-reinstall "{cached_whl[0]}"
    elif os.path.exists(sub_path):
        print(f"🔨 Đang biên dịch {sub_name} và lưu wheel vào Drive (Chỉ thực hiện 1 lần)...")
        !pip wheel --no-build-isolation "{sub_path}" -w "{WHEELS_CACHE_DIR}"
        new_whl = glob.glob(os.path.join(WHEELS_CACHE_DIR, f"{clean_name}*.whl"))
        if new_whl:
            !pip install --no-deps --force-reinstall "{new_whl[0]}"
        else:
            !pip install -e "{sub_path}"
    else:
        print(f"⚠️ Không tìm thấy thư mục {sub_path}")

print("\n🔍 Kiểm tra các module CUDA đã cài đặt:")
!python -c "import diff_gaussian_rasterization_fastgs; import simple_knn; import fused_ssim; print('✅ All submodules loaded successfully in PyTorch!')"

📁 Thư mục Cache Wheels : /content/drive/MyDrive/Thesis/wheels_cache
🔨 Thư mục Submodules gốc: /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs/submodules
⚡ Tìm thấy pre-built wheel cho 'diff-gaussian-rasterization_fastgs': diff_gaussian_rasterization_fastgs-0.0.0-cp312-cp312-linux_x86_64.whl
   🚀 Cài đặt wheel siêu tốc...
Processing ./drive/MyDrive/Thesis/wheels_cache/diff_gaussian_rasterization_fastgs-0.0.0-cp312-cp312-linux_x86_64.whl
  Attempting uninstall: diff-gaussian-rasterization-fastgs
    Found existing installation: diff_gaussian_rasterization_fastgs 0.0.0
    Uninstalling diff_gaussian_rasterization_fastgs-0.0.0:
      Successfully uninstalled diff_gaussian_rasterization_fastgs-0.0.0
⚡ Tìm thấy pre-built wheel cho 'simple-knn': simple_knn-0.0.0-cp312-cp312-linux_x86_64.whl
   🚀 Cài đặt wheel siêu tốc...
Processing ./drive/MyDrive/Thesis/wheels_cache/simple_knn-0.0.0-cp312-cp312-linux_x86_64.whl
  Attempting uninstall: simple-knn
    Found existing installation: sim

## 📥 Bước 4: Chuẩn Bị Dataset Scene `teapot` [Hình 2] & Tối Ưu Bộ Nhớ

> Tự động kiểm tra dataset `teapot` trên MyDrive, tạo symlink và nạp các bản vá tối ưu bộ nhớ (chống tràn RAM OOM / Kill 137).

In [ ]:
# ── Kiểm tra & Tải Dataset scene 'teapot' (Chỉ làm 1 lần) ─────────────────────
import os
import shutil

has_transforms = os.path.exists(os.path.join(DATASET_TEAPOT_DIR, "transforms_train.json")) or \
                 os.path.exists(os.path.join(DATASET_TEAPOT_DIR, "transforms.json")) or \
                 os.path.exists(os.path.join(DATASET_TEAPOT_DIR, "transforms_test.json"))

if has_transforms:
    print(f"✅ Dataset scene '{SCENE_NAME}' đã có sẵn trên MyDrive tại: {DATASET_TEAPOT_DIR}")
else:
    print(f"🔍 Chưa tìm thấy scene '{SCENE_NAME}' tại {DATASET_TEAPOT_DIR}. Đang tải từ Drive API...")
    items = list_drive_folder_items(GDRIVE_DATASET_FOLDER_ID)
    teapot_item = next((i for i in items if i['name'] == SCENE_NAME and i['mimeType'] == 'application/vnd.google-apps.folder'), None)

    if teapot_item:
        print(f"📂 Tìm thấy folder '{SCENE_NAME}' (ID: {teapot_item['id']}). Đang tải về MyDrive...")
        download_drive_folder_recursive(teapot_item['id'], DATASET_TEAPOT_DIR)
        print(f"✅ Đã lưu dataset scene '{SCENE_NAME}' vào MyDrive: {DATASET_TEAPOT_DIR}")
    else:
        print(f"📥 Tải toàn bộ dataset từ Folder ID: {GDRIVE_DATASET_FOLDER_ID}...")
        download_drive_folder_recursive(GDRIVE_DATASET_FOLDER_ID, GDRIVE_DATASET_DIR)

# ── Thiết lập liên kết datasets/Anisotropic-Synthesis/teapot bên trong repo ───────
for alias in ["Anisotropic-Synthesis", "Anisotropic-Synthetic-Dataset", "synthetic_specular"]:
    alias_dir = os.path.join(REPO_DIR, "datasets", alias)
    os.makedirs(alias_dir, exist_ok=True)
    alias_teapot = os.path.join(alias_dir, SCENE_NAME)
    if not os.path.exists(alias_teapot):
        try:
            os.symlink(DATASET_TEAPOT_DIR, alias_teapot)
            print(f"🔗 Tạo Symlink: {alias_teapot} -> {DATASET_TEAPOT_DIR}")
        except Exception:
            shutil.copytree(DATASET_TEAPOT_DIR, alias_teapot, dirs_exist_ok=True)
            print(f"📁 Đồng bộ thư mục: {alias_teapot}")

# ── Tối ưu extract_reflection_prior.py chống tràn RAM (OOM / Kill 137) ────────
extract_py_path = os.path.join(REPO_DIR, "extract_reflection_prior.py")
memory_safe_extractor = '''# ============================================================
# Extract Reflection Prior
# ============================================================

import os
import time
import json
import imageio
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from argparse import ArgumentParser
from arguments import ModelParams
from utils.general_utils import safe_state

def _soft_step(x, edge, scale=12.0):
    x_clip = np.clip(x - edge, -2.0, 2.0)
    return 1.0 / (1.0 + np.exp(-scale * x_clip))

def _normalize_score(score, eps=1e-6):
    smin, smax = np.min(score), np.max(score)
    if (smax - smin) < eps:
        return np.zeros_like(score, dtype=np.float32)
    return np.clip((score - smin) / (smax - smin + eps), 0.0, 1.0).astype(np.float32)

def _box_blur_2d(img, radius=2):
    if radius <= 0:
        return img.astype(np.float32)
    try:
        from scipy.ndimage import uniform_filter
        return uniform_filter(img.astype(np.float32), size=2 * radius + 1, mode="reflect")
    except Exception:
        return img.astype(np.float32)

def postprocess_score(score, gamma=1.0, quantile=0.0, smooth_radius=0):
    score = np.clip(score, 0.0, 1.0).astype(np.float32)
    if quantile > 0.0:
        pos_mask = score > 0.0
        if np.any(pos_mask):
            cutoff = np.quantile(score[pos_mask], quantile)
            score = np.where(score >= cutoff, (score - cutoff) / max(1e-6, 1.0 - cutoff), 0.0)
            score = np.clip(score, 0.0, 1.0).astype(np.float32)
    if smooth_radius > 0:
        score = _box_blur_2d(score, radius=smooth_radius)
        score = np.clip(score, 0.0, 1.0).astype(np.float32)
    if abs(gamma - 1.0) > 1e-3:
        score = np.power(np.clip(score, 0.0, 1.0), gamma)
    return score.astype(np.float32)

def tan_ikeuchi_score(img01, thresh=0.35, bright=0.60):
    Imax = np.max(img01, axis=-1)
    Imin = np.min(img01, axis=-1)
    spec_score = 3.0 * Imin - Imax
    pos_mask = spec_score > 0
    if np.any(pos_mask):
        s_min = np.min(spec_score[pos_mask])
        s_max = np.max(spec_score[pos_mask])
        if s_max > s_min:
            spec_score = np.where(pos_mask, (spec_score - s_min) / (s_max - s_min), 0.0)
        else:
            spec_score = np.zeros_like(spec_score)
    else:
        spec_score = np.zeros_like(spec_score)
    spec_score = np.where(Imax > bright, spec_score, 0.0)
    spec_score = np.where(spec_score > thresh, spec_score, 0.0)
    return spec_score

def shafer_klinker_score(img01, sk_intensity=0.7, sk_saturation=0.2):
    Imax = np.max(img01, axis=-1)
    Imin = np.min(img01, axis=-1)
    saturation = (Imax - Imin) / (Imax + 1e-6)
    is_spec = (Imax >= sk_intensity) & (saturation <= sk_saturation)
    intensity_term = (Imax - sk_intensity) / max(1e-6, 1.0 - sk_intensity)
    saturation_term = (sk_saturation - saturation) / max(1e-6, sk_saturation)
    score = np.where(is_spec, 0.5 * (intensity_term + saturation_term), 0.0)
    return _normalize_score(score)

def hybrid_confidence_score(img01, ti_thresh=0.35, ti_bright=0.60, sk_intensity=0.7, sk_saturation=0.2):
    Imax = np.max(img01, axis=-1)
    Imin = np.min(img01, axis=-1)
    saturation = (Imax - Imin) / (Imax + 1e-6)
    ti_raw = np.clip((3.0 * Imin - Imax), 0.0, None)
    ti_soft = _soft_step(ti_raw, ti_thresh) * _soft_step(Imax, ti_bright)
    shafer_soft = _soft_step(Imax, sk_intensity) * _soft_step(sk_saturation - saturation, 0.0)
    local_mean = _box_blur_2d(Imax.astype(np.float32), radius=3)
    local_highlight = _normalize_score(np.clip(Imax - local_mean, 0.0, None))
    gray_bright = _normalize_score(Imax * (1.0 - saturation))
    score = 0.35 * ti_soft + 0.35 * shafer_soft + 0.20 * gray_bright + 0.10 * local_highlight
    return _normalize_score(score)

def get_camera_manifest(dataset):
    source_path = dataset.source_path
    tf_candidates = ["transforms_train.json", "transforms.json"]
    for tf in tf_candidates:
        tf_path = os.path.join(source_path, tf)
        if os.path.exists(tf_path):
            with open(tf_path, "r") as f:
                data = json.load(f)
            frames = data.get("frames", [])
            manifest = []
            for frame in frames:
                rel = frame["file_path"]
                if not rel.endswith(".png") and not rel.endswith(".jpg"): rel += ".png"
                full_path = os.path.join(source_path, rel)
                if not os.path.exists(full_path): full_path = os.path.join(source_path, rel.lstrip("./"))
                name = Path(rel).stem
                manifest.append((name, full_path))
            if manifest: return manifest
    img_dir = os.path.join(source_path, "images")
    if os.path.isdir(img_dir):
        files = sorted(os.listdir(img_dir))
        return [(Path(f).stem, os.path.join(img_dir, f)) for f in files if f.lower().endswith(('.png', '.jpg'))]
    return []

def extract_priors(dataset, args):
    start_time = time.time()
    camera_manifest = get_camera_manifest(dataset)
    print(f"Loaded {len(camera_manifest)} training cameras (streaming memory-safe mode).")
    save_dir = os.path.join(dataset.source_path, "reflection_prior")
    os.makedirs(save_dir, exist_ok=True)
    print(f"Output directory: {save_dir}")
    progress_bar = tqdm(camera_manifest, desc=f"Extracting Priors ({args.ref_prior_method})")
    for ref_image_name, image_path in progress_bar:
        if image_path and os.path.isfile(image_path):
            rgba = np.asarray(Image.open(image_path).convert("RGBA"), dtype=np.float32) / 255.0
            img01 = rgba[..., :3]
            foreground_alpha = rgba[..., 3]
        else:
            continue
        if args.ref_prior_method == "tan":
            final_score = tan_ikeuchi_score(img01, thresh=args.ti_thresh, bright=args.ti_bright)
        elif args.ref_prior_method == "shafer":
            final_score = shafer_klinker_score(img01, sk_intensity=args.sk_intensity, sk_saturation=args.sk_saturation)
        elif args.ref_prior_method == "hybrid":
            final_score = hybrid_confidence_score(img01, ti_thresh=args.ti_thresh, ti_bright=args.ti_bright, sk_intensity=args.sk_intensity, sk_saturation=args.sk_saturation)
        else:
            raise ValueError(f"Unknown ref_prior_method: {args.ref_prior_method}")
        final_score = final_score * foreground_alpha
        ref_score = _normalize_score(final_score)
        ref_conf = postprocess_score(final_score, gamma=args.ref_conf_gamma, quantile=args.ref_conf_quantile, smooth_radius=args.ref_conf_smooth_radius)
        score_img = (ref_score * 255).astype(np.uint8)
        conf_img = (ref_conf * 255).astype(np.uint8)
        save_path = os.path.join(save_dir, f"{ref_image_name}_ref_score.png")
        conf_path = os.path.join(save_dir, f"{ref_image_name}_ref_conf.png")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        imageio.imwrite(save_path, score_img)
        imageio.imwrite(conf_path, conf_img)
    end_time = time.time()
    print(f"Prior extraction complete in {end_time - start_time:.2f} seconds!")

if __name__ == "__main__":
    parser = ArgumentParser(description="Extract Reflection Prior")
    lp = ModelParams(parser)
    parser.add_argument("--ref_prior_method", type=str, default="tan", choices=["tan", "shafer", "hybrid"])
    parser.add_argument("--ti_thresh", type=float, default=0.35)
    parser.add_argument("--ti_bright", type=float, default=0.6)
    parser.add_argument("--sk_intensity", type=float, default=0.7)
    parser.add_argument("--sk_saturation", type=float, default=0.2)
    parser.add_argument("--ref_conf_gamma", type=float, default=1.0)
    parser.add_argument("--ref_conf_quantile", type=float, default=0.0)
    parser.add_argument("--ref_conf_smooth_radius", type=int, default=0)
    args = parser.parse_args()
    safe_state(False)
    extract_priors(lp.extract(args), args)
'''
with open(extract_py_path, 'w', encoding='utf-8') as f:
    f.write(memory_safe_extractor)
print(f"⚡ Đã áp dụng streaming memory-safe cho {extract_py_path} (Kế thừa 100% code mới & chống tràn RAM OOM Kill 137)!")

# ── Tối ưu scene/__init__.py và render.py chống tràn RAM & đồng bộ thiết bị ──
scene_init_path = os.path.join(REPO_DIR, "scene", "__init__.py")
if os.path.exists(scene_init_path):
    with open(scene_init_path, "r", encoding="utf-8") as f:
        s_code = f.read()
    s_code = s_code.replace(
        'from scene.dataset_readers import sceneLoadTypeCallbacks, readCamerasFromJSONFile',
        'from scene.dataset_readers import sceneLoadTypeCallbacks\ntry:\n    from scene.dataset_readers import readCamerasFromJSONFile\nexcept ImportError:\n    readCamerasFromJSONFile = None'
    )
    s_code = s_code.replace(
        'def save(self, iteration):',
        'def save(self, iteration, save_asg=True, **kwargs):'
    )
    if "getattr(args, \"skip_train\", False)" not in s_code:
        s_code = s_code.replace(
            'print("Loading Training Cameras")\n            self.train_cameras[scale] = cameraList_from_camInfos(\n                scene_info.train_cameras,\n                scale,\n                args\n            )',
            'if not getattr(args, "skip_train", False):\n                print("Loading Training Cameras")\n                self.train_cameras[scale] = cameraList_from_camInfos(scene_info.train_cameras, scale, args)\n            else:\n                self.train_cameras[scale] = []'
        )
        s_code = s_code.replace(
            'print("Loading Test Cameras")\n            self.test_cameras[scale] = cameraList_from_camInfos(\n                scene_info.test_cameras,\n                scale,\n                args\n            )',
            'if not getattr(args, "skip_test", False):\n                print("Loading Test Cameras")\n                self.test_cameras[scale] = cameraList_from_camInfos(scene_info.test_cameras, scale, args)\n            else:\n                self.test_cameras[scale] = []\n        scene_info.train_cameras.clear()\n        scene_info.test_cameras.clear()'
        )
    with open(scene_init_path, "w", encoding="utf-8") as f:
        f.write(s_code)
    print("⚡ Đã tối ưu hóa scene/__init__.py (Khắc phục lỗi save_asg, ImportError & chống tràn RAM)!")

render_py_path = os.path.join(REPO_DIR, "render.py")
if os.path.exists(render_py_path):
    with open(render_py_path, "r", encoding="utf-8") as f:
        r_code = f.read()
    r_code = r_code.replace(
        "gt = view.original_image[0:3, :, :]",
        "gt = view.original_image[0:3, :, :].to(rendering.device)"
    )
    if "dataset.skip_train = skip_train" not in r_code:
        r_code = r_code.replace(
            "gaussians = GaussianModel(dataset.sh_degree, dataset.asg_degree)\n        scene = Scene(dataset, gaussians",
            "dataset.skip_train = skip_train\n        dataset.skip_test = skip_test\n        gaussians = GaussianModel(dataset.sh_degree, dataset.asg_degree)\n        scene = Scene(dataset, gaussians"
        )
    with open(render_py_path, "w", encoding="utf-8") as f:
        f.write(r_code)
    print("⚡ Đã tối ưu hóa render.py (Đồng bộ Device CUDA/CPU & Truyền cờ skip_train)!")

print("\n🔍 Kiểm tra đường dẫn dataset trong repo:")
!ls -la "{REPO_DIR}/datasets/Anisotropic-Synthesis/teapot"

✅ Dataset scene 'teapot' đã có sẵn trên MyDrive tại: /content/drive/MyDrive/Anisotropic-Synthetic-Dataset/teapot
⚡ Đã áp dụng streaming memory-safe cho /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs/extract_reflection_prior.py (Kế thừa 100% code mới & chống tràn RAM OOM Kill 137)!
⚡ Đã tối ưu hóa scene/__init__.py (Khắc phục lỗi save_asg, ImportError & chống tràn RAM)!
⚡ Đã tối ưu hóa render.py (Đồng bộ Device CUDA/CPU & Truyền cờ skip_train)!

🔍 Kiểm tra đường dẫn dataset trong repo:
lrw------- 1 root root 0 Aug 20 07:24 /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs/datasets/Anisotropic-Synthesis/teapot -> /content/drive/.shortcut-targets-by-id/1vJaYQB8jLHt7TcHy1u6o9Ergey5yZ4l7/Anisotropic-Synthetic-Dataset/teapot


## 🏃‍♂️ Bước 5: RUN 1 — Baseline Spec-FastGS (Đầy Đủ 3 Cờ)

> **Mô tả cấu hình**: Baseline hoàn chỉnh của Spec-FastGS với đầy đủ 3 cơ chế hướng dẫn phản xạ.
> **Thư mục Output**: `output/anisotropic_synthetic/teapot`
> **Cơ chế Auto-Skip**: Nếu đã tồn tại `results.json` $\rightarrow$ Bỏ qua trong 0.1s!

In [ ]:
# # ── Tạo Script Chạy Cho RUN 1 (exec_run1_teapot.sh) ─────────────
# import os

# run_sh_path = os.path.join(REPO_DIR, "exec_run1_teapot.sh")
# run_script_content = """#!/bin/bash
# set -e
# export CUDA_VISIBLE_DEVICES=0

# echo "======================================================================="
# echo "🚀 BẮT ĐẦU RUN 1: Baseline Spec-FastGS (Đầy Đủ 3 Cờ)"
# echo "======================================================================="

# OUTPUT_PATH="output/anisotropic_synthetic/teapot"

# # 1. Kiểm tra Auto-Skip: Nếu đã có results.json thì bỏ qua 100%
# if [ -f "$OUTPUT_PATH/results.json" ]; then
#     echo "✅ RUN 1 đã hoàn thành đầy đủ trước đó (đã có results.json). Bỏ qua toàn bộ bước này!"
#     exit 0
# fi

# # 2. Trích xuất Reflection Prior (nếu chưa có)
# if [ ! -d "datasets/Anisotropic-Synthesis/teapot/reflection_prior" ] || [ $(ls -1 datasets/Anisotropic-Synthesis/teapot/reflection_prior | wc -l) -eq 0 ]; then
#     python extract_reflection_prior.py \
#         -s datasets/Anisotropic-Synthesis/teapot \
#         --eval \
#         --white_background \
#         --data_device cpu \
#         --ref_prior_method tan \
#         --ti_thresh 0.35 \
#         --ti_bright 0.60
# else
#     echo "⚡ Reflection Prior đã tồn tại. Bỏ qua bước trích xuất prior!"
# fi

# # 3. Huấn luyện (Nếu đã có checkpoint 30k thì bỏ qua train.py)
# if [ -f "$OUTPUT_PATH/point_cloud/iteration_30000/point_cloud.ply" ]; then
#     echo "⚡ Checkpoint 30,000 iterations đã hoàn thành trước đó! Chuyển thẳng sang bước Render..."
# else
#     python train.py \
#         -s datasets/Anisotropic-Synthesis/teapot \
#         -m $OUTPUT_PATH \
#         --eval \
#         --white_background \
#         --asg_degree 24 \
#         --densification_interval 500 \
#         --densification_refscore_interval 500 \
#         --num_score_cameras 10 \
#         --optimizer_type default \
#         --use_ref_score \
#         --use_adaptive_prior \
#         --use_reflection_view_sampling
#         # --disable_multiview_contribution
# fi

# # 4. Kết xuất ảnh (Render)
# python render.py -s datasets/Anisotropic-Synthesis/teapot -m $OUTPUT_PATH --iteration 30000 --skip_train --data_device cpu

# # 5. Tính toán Metrics
# python metrics.py -m $OUTPUT_PATH

# echo "🎉 Hoàn tất RUN 1 (teapot) thành công!"
# """

# with open(run_sh_path, 'w', encoding='utf-8') as f:
#     f.write(run_script_content)

# !chmod +x "{run_sh_path}"
# print(f"📜 Đã khởi tạo script chạy RUN 1: {run_sh_path}")


In [ ]:
# %%bash
# # ── Thực thi RUN 1: Baseline Spec-FastGS (Đầy Đủ 3 Cờ) ───────────────────────────────────────
# set -e
# export CUDA_HOME=/usr/local/cuda
# export PATH=$CUDA_HOME/bin:$PATH
# TORCH_LIB=$(python3 -c 'import torch, os; print(os.path.join(os.path.dirname(torch.__file__), "lib"))' 2>/dev/null || echo "")
# export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$TORCH_LIB:$LD_LIBRARY_PATH
# export CUDA_VISIBLE_DEVICES=0

# MYDRIVE_ROOT="/content/drive/MyDrive"
# REPO_DIR="$MYDRIVE_ROOT/Thesis/source/20082026/spec-fastgs"
# cd "$REPO_DIR"

# bash exec_run1_teapot.sh


## ☁️ Bước 6: Đẩy Kết Quả RUN 1 Lên Thư Mục Google Drive [Hình 3] (`teapot`)

> Upload toàn bộ kết quả lên Google Drive Folder ID `1FZdVaZmY9VsFMGXWc5107z4pGz-cTyCh` với tên folder là `teapot` (**loại trừ hoàn toàn thư mục `test/`**).

In [ ]:
# # ── Lưu & Đẩy Kết Quả RUN 1 Lên Drive (Loại trừ folder 'test') ─────────────────
# import os
# import shutil

# RUN_FOLDER_NAME = "teapot"
# run_teapot_output = os.path.join(REPO_DIR, "output", "anisotropic_synthetic", RUN_FOLDER_NAME)
# if not os.path.exists(run_teapot_output):
#     for root, dirs, files in os.walk(os.path.join(REPO_DIR, "output")):
#         if RUN_FOLDER_NAME in root and ("point_cloud" in dirs or "cameras.json" in files or "results.json" in files):
#             run_teapot_output = root
#             break

# print(f"📂 Thư mục output RUN 1: {run_teapot_output}")

# # 1. Đẩy trực tiếp qua Google Drive API vào Folder ID ở Hình 3
# upload_local_folder_to_drive(
#     local_dir=run_teapot_output,
#     parent_folder_id=GDRIVE_RESULTS_FOLDER_ID,
#     folder_name=RUN_FOLDER_NAME,
#     exclude_names=['test']
# )

# # 2. Đồng thời lưu bản sao vào MyDrive
# drive_dest = os.path.join(GDRIVE_RESULTS_DIR, RUN_FOLDER_NAME)
# try:
#     os.makedirs(drive_dest, exist_ok=True)
#     for root, dirs, files in os.walk(run_teapot_output):
#         if 'test' in [d.lower() for d in dirs]: dirs.remove('test')
#         rel = os.path.relpath(root, run_teapot_output)
#         dst_d = os.path.join(drive_dest, rel) if rel != '.' else drive_dest
#         os.makedirs(dst_d, exist_ok=True)
#         for f in files:
#             shutil.copy2(os.path.join(root, f), os.path.join(dst_d, f))
#     print(f"✅ Đã đồng bộ thêm bản sao vào: {drive_dest}")
# except Exception as e:
#     print(f"ℹ️ Ghi chú MyDrive path: {e}")

# print(f"\n🎉 Hoàn tất đẩy kết quả '{RUN_FOLDER_NAME}' lên Google Drive!")


## 🏃‍♂️ Bước 7: RUN 2 — Ablation: Disable Multiview Contribution

> **Mô tả cấu hình**: Bật cờ `--disable_multiview_contribution` để đánh giá tác động khi tắt đóng góp đa góc nhìn.
> **Thư mục Output**: `output/anisotropic_synthetic/teapot_disable_multiview`
> **Cơ chế Auto-Skip**: Nếu đã tồn tại `results.json` $\rightarrow$ Bỏ qua trong 0.1s!

In [ ]:
# # ── Tạo Script Chạy Cho RUN 2 (exec_run2_teapot_teapot_disable_multiview.sh) ─────────────
# import os

# run_sh_path = os.path.join(REPO_DIR, "exec_run2_teapot_teapot_disable_multiview.sh")
# run_script_content = """#!/bin/bash
# set -e
# export CUDA_VISIBLE_DEVICES=0

# echo "======================================================================="
# echo "🚀 BẮT ĐẦU RUN 2: Ablation: Disable Multiview Contribution"
# echo "======================================================================="

# OUTPUT_PATH="output/anisotropic_synthetic/teapot_disable_multiview"

# # 1. Kiểm tra Auto-Skip: Nếu đã có results.json thì bỏ qua 100%
# if [ -f "$OUTPUT_PATH/results.json" ]; then
#     echo "✅ RUN 2 đã hoàn thành đầy đủ trước đó (đã có results.json). Bỏ qua toàn bộ bước này!"
#     exit 0
# fi

# # 2. Trích xuất Reflection Prior (nếu chưa có)
# if [ ! -d "datasets/Anisotropic-Synthesis/teapot/reflection_prior" ] || [ $(ls -1 datasets/Anisotropic-Synthesis/teapot/reflection_prior | wc -l) -eq 0 ]; then
#     python extract_reflection_prior.py \
#         -s datasets/Anisotropic-Synthesis/teapot \
#         --eval \
#         --white_background \
#         --data_device cpu \
#         --ref_prior_method tan \
#         --ti_thresh 0.35 \
#         --ti_bright 0.60
# else
#     echo "⚡ Reflection Prior đã tồn tại. Bỏ qua bước trích xuất prior!"
# fi

# # 3. Huấn luyện (Nếu đã có checkpoint 30k thì bỏ qua train.py)
# if [ -f "$OUTPUT_PATH/point_cloud/iteration_30000/point_cloud.ply" ]; then
#     echo "⚡ Checkpoint 30,000 iterations đã hoàn thành trước đó! Chuyển thẳng sang bước Render..."
# else
#     python train.py \
#         -s datasets/Anisotropic-Synthesis/teapot \
#         -m $OUTPUT_PATH \
#         --eval \
#         --white_background \
#         --asg_degree 24 \
#         --densification_interval 500 \
#         --densification_refscore_interval 500 \
#         --num_score_cameras 10 \
#         --optimizer_type default \
#         --use_ref_score \
#         --use_adaptive_prior \
#         --use_reflection_view_sampling \
#         --disable_multiview_contribution
# fi

# # 4. Kết xuất ảnh (Render)
# python render.py -s datasets/Anisotropic-Synthesis/teapot -m $OUTPUT_PATH --iteration 30000 --skip_train --data_device cpu

# # 5. Tính toán Metrics
# python metrics.py -m $OUTPUT_PATH

# echo "🎉 Hoàn tất RUN 2 (teapot_disable_multiview) thành công!"
# """

# with open(run_sh_path, 'w', encoding='utf-8') as f:
#     f.write(run_script_content)

# !chmod +x "{run_sh_path}"
# print(f"📜 Đã khởi tạo script chạy RUN 2: {run_sh_path}")


In [ ]:
# %%bash
# # ── Thực thi RUN 2: Ablation: Disable Multiview Contribution ───────────────────────────────────────
# set -e
# export CUDA_HOME=/usr/local/cuda
# export PATH=$CUDA_HOME/bin:$PATH
# TORCH_LIB=$(python3 -c 'import torch, os; print(os.path.join(os.path.dirname(torch.__file__), "lib"))' 2>/dev/null || echo "")
# export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$TORCH_LIB:$LD_LIBRARY_PATH
# export CUDA_VISIBLE_DEVICES=0

# MYDRIVE_ROOT="/content/drive/MyDrive"
# REPO_DIR="$MYDRIVE_ROOT/Thesis/source/20082026/spec-fastgs"
# cd "$REPO_DIR"

# bash exec_run2_teapot_teapot_disable_multiview.sh


## ☁️ Bước 8: Đẩy Kết Quả RUN 2 Lên Thư Mục Google Drive [Hình 3] (`teapot_disable_multiview`)

> Upload toàn bộ kết quả lên Google Drive Folder ID `1FZdVaZmY9VsFMGXWc5107z4pGz-cTyCh` với tên folder là `teapot_disable_multiview` (**loại trừ hoàn toàn thư mục `test/`**).

In [ ]:
# # ── Lưu & Đẩy Kết Quả RUN 2 Lên Drive (Loại trừ folder 'test') ─────────────────
# import os
# import shutil

# RUN_FOLDER_NAME = "teapot_disable_multiview"
# run_teapot_output = os.path.join(REPO_DIR, "output", "anisotropic_synthetic", RUN_FOLDER_NAME)
# if not os.path.exists(run_teapot_output):
#     for root, dirs, files in os.walk(os.path.join(REPO_DIR, "output")):
#         if RUN_FOLDER_NAME in root and ("point_cloud" in dirs or "cameras.json" in files or "results.json" in files):
#             run_teapot_output = root
#             break

# print(f"📂 Thư mục output RUN 2: {run_teapot_output}")

# # 1. Đẩy trực tiếp qua Google Drive API vào Folder ID ở Hình 3
# upload_local_folder_to_drive(
#     local_dir=run_teapot_output,
#     parent_folder_id=GDRIVE_RESULTS_FOLDER_ID,
#     folder_name=RUN_FOLDER_NAME,
#     exclude_names=['test']
# )

# # 2. Đồng thời lưu bản sao vào MyDrive
# drive_dest = os.path.join(GDRIVE_RESULTS_DIR, RUN_FOLDER_NAME)
# try:
#     os.makedirs(drive_dest, exist_ok=True)
#     for root, dirs, files in os.walk(run_teapot_output):
#         if 'test' in [d.lower() for d in dirs]: dirs.remove('test')
#         rel = os.path.relpath(root, run_teapot_output)
#         dst_d = os.path.join(drive_dest, rel) if rel != '.' else drive_dest
#         os.makedirs(dst_d, exist_ok=True)
#         for f in files:
#             shutil.copy2(os.path.join(root, f), os.path.join(dst_d, f))
#     print(f"✅ Đã đồng bộ thêm bản sao vào: {drive_dest}")
# except Exception as e:
#     print(f"ℹ️ Ghi chú MyDrive path: {e}")

# print(f"\n🎉 Hoàn tất đẩy kết quả '{RUN_FOLDER_NAME}' lên Google Drive!")


## 🏃‍♂️ Bước 9: RUN 3 — Ablation: Chỉ Dùng --use_ref_score

> **Mô tả cấu hình**: Chỉ sử dụng Reflection Score đơn thuần, tắt Adaptive Prior và Reflection View Sampling.
> **Thư mục Output**: `output/anisotropic_synthetic/teapot_only_ref_score`
> **Cơ chế Auto-Skip**: Nếu đã tồn tại `results.json` $\rightarrow$ Bỏ qua trong 0.1s!

In [ ]:
# # ── Tạo Script Chạy Cho RUN 3 (exec_run3_teapot_teapot_only_ref_score.sh) ─────────────
# import os

# run_sh_path = os.path.join(REPO_DIR, "exec_run3_teapot_teapot_only_ref_score.sh")
# run_script_content = """#!/bin/bash
# set -e
# export CUDA_VISIBLE_DEVICES=0

# echo "======================================================================="
# echo "🚀 BẮT ĐẦU RUN 3: Ablation: Chỉ Dùng --use_ref_score"
# echo "======================================================================="

# OUTPUT_PATH="output/anisotropic_synthetic/teapot_only_ref_score"

# # 1. Kiểm tra Auto-Skip: Nếu đã có results.json thì bỏ qua 100%
# if [ -f "$OUTPUT_PATH/results.json" ]; then
#     echo "✅ RUN 3 đã hoàn thành đầy đủ trước đó (đã có results.json). Bỏ qua toàn bộ bước này!"
#     exit 0
# fi

# # 2. Trích xuất Reflection Prior (nếu chưa có)
# if [ ! -d "datasets/Anisotropic-Synthesis/teapot/reflection_prior" ] || [ $(ls -1 datasets/Anisotropic-Synthesis/teapot/reflection_prior | wc -l) -eq 0 ]; then
#     python extract_reflection_prior.py \
#         -s datasets/Anisotropic-Synthesis/teapot \
#         --eval \
#         --white_background \
#         --data_device cpu \
#         --ref_prior_method tan \
#         --ti_thresh 0.35 \
#         --ti_bright 0.60
# else
#     echo "⚡ Reflection Prior đã tồn tại. Bỏ qua bước trích xuất prior!"
# fi

# # 3. Huấn luyện (Nếu đã có checkpoint 30k thì bỏ qua train.py)
# if [ -f "$OUTPUT_PATH/point_cloud/iteration_30000/point_cloud.ply" ]; then
#     echo "⚡ Checkpoint 30,000 iterations đã hoàn thành trước đó! Chuyển thẳng sang bước Render..."
# else
#     python train.py \
#         -s datasets/Anisotropic-Synthesis/teapot \
#         -m $OUTPUT_PATH \
#         --eval \
#         --white_background \
#         --asg_degree 24 \
#         --densification_interval 500 \
#         --densification_refscore_interval 500 \
#         --num_score_cameras 10 \
#         --optimizer_type default \
#         --use_ref_score
#         # --use_adaptive_prior
#         # --use_reflection_view_sampling
#         # --disable_multiview_contribution
# fi

# # 4. Kết xuất ảnh (Render)
# python render.py -s datasets/Anisotropic-Synthesis/teapot -m $OUTPUT_PATH --iteration 30000 --skip_train --data_device cpu

# # 5. Tính toán Metrics
# python metrics.py -m $OUTPUT_PATH

# echo "🎉 Hoàn tất RUN 3 (teapot_only_ref_score) thành công!"
# """

# with open(run_sh_path, 'w', encoding='utf-8') as f:
#     f.write(run_script_content)

# !chmod +x "{run_sh_path}"
# print(f"📜 Đã khởi tạo script chạy RUN 3: {run_sh_path}")


📜 Đã khởi tạo script chạy RUN 3: /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs/exec_run3_teapot_teapot_only_ref_score.sh


In [ ]:
# %%bash
# # ── Thực thi RUN 3: Ablation: Chỉ Dùng --use_ref_score ───────────────────────────────────────
# set -e
# export CUDA_HOME=/usr/local/cuda
# export PATH=$CUDA_HOME/bin:$PATH
# TORCH_LIB=$(python3 -c 'import torch, os; print(os.path.join(os.path.dirname(torch.__file__), "lib"))' 2>/dev/null || echo "")
# export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$TORCH_LIB:$LD_LIBRARY_PATH
# export CUDA_VISIBLE_DEVICES=0

# MYDRIVE_ROOT="/content/drive/MyDrive"
# REPO_DIR="$MYDRIVE_ROOT/Thesis/source/20082026/spec-fastgs"
# cd "$REPO_DIR"

# bash exec_run3_teapot_teapot_only_ref_score.sh


🚀 BẮT ĐẦU RUN 3: Ablation: Chỉ Dùng --use_ref_score
⚡ Reflection Prior đã tồn tại. Bỏ qua bước trích xuất prior!
Output folder already exists and is not empty. Moving old run to: output/anisotropic_synthetic/backups/teapot_only_ref_score/unknown_20260820_155312 [20/08 15:55:12]
Output folder: output/anisotropic_synthetic/teapot_only_ref_score [20/08 15:55:12]
[SH-ASG Ablation] use_asg=True [20/08 15:55:12]
[FastGS Ablation] VCD=True, VCP=True, CompactBox=True, beta=0.5 [20/08 15:55:12]
Detected Blender dataset (transforms_train.json) [20/08 15:55:12]
Reading Training Transforms [20/08 15:55:12]
Reading Test Transforms [20/08 15:55:48]
Loading Training Cameras [20/08 15:55:53]
Loading Test Cameras [20/08 15:56:01]
Number of points at initialisation :  100000 [20/08 15:56:02]
[Auto Budget] Ref Score cap: 1,000,000 Gaussians (initial=100,000, multiplier=10.0) [20/08 15:56:02]
Loading Reflection Priors... [20/08 15:56:03]
Loaded 600 reflection priors; missing 0. [20/08 15:56:14]
Model chec

2026-08-20 15:55:04.359408: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
/content/drive/.shortcut-targets-by-id/1f8jbBEpIwgX61UWWi0ezERkYdRKYGReJ/Thesis/source/20082026/spec-fastgs/train.py:535: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  prior_img = imageio.imread(path)
test Gaussian heatmaps: 100%|██████████| 100/100 [00:16<00:00,  6.20it/s]
fatal: not a git repository (or any parent up to mount

## ☁️ Bước 10: Đẩy Kết Quả RUN 3 Lên Thư Mục Google Drive [Hình 3] (`teapot_only_ref_score`)

> Upload toàn bộ kết quả lên Google Drive Folder ID `1FZdVaZmY9VsFMGXWc5107z4pGz-cTyCh` với tên folder là `teapot_only_ref_score` (**loại trừ hoàn toàn thư mục `test/`**).

In [ ]:
# # ── Lưu & Đẩy Kết Quả RUN 3 Lên Drive (Loại trừ folder 'test') ─────────────────
# import os
# import shutil

# RUN_FOLDER_NAME = "teapot_only_ref_score"
# run_teapot_output = os.path.join(REPO_DIR, "output", "anisotropic_synthetic", RUN_FOLDER_NAME)
# if not os.path.exists(run_teapot_output):
#     for root, dirs, files in os.walk(os.path.join(REPO_DIR, "output")):
#         if RUN_FOLDER_NAME in root and ("point_cloud" in dirs or "cameras.json" in files or "results.json" in files):
#             run_teapot_output = root
#             break

# print(f"📂 Thư mục output RUN 3: {run_teapot_output}")

# # 1. Đẩy trực tiếp qua Google Drive API vào Folder ID ở Hình 3
# upload_local_folder_to_drive(
#     local_dir=run_teapot_output,
#     parent_folder_id=GDRIVE_RESULTS_FOLDER_ID,
#     folder_name=RUN_FOLDER_NAME,
#     exclude_names=['test']
# )

# # 2. Đồng thời lưu bản sao vào MyDrive
# drive_dest = os.path.join(GDRIVE_RESULTS_DIR, RUN_FOLDER_NAME)
# try:
#     os.makedirs(drive_dest, exist_ok=True)
#     for root, dirs, files in os.walk(run_teapot_output):
#         if 'test' in [d.lower() for d in dirs]: dirs.remove('test')
#         rel = os.path.relpath(root, run_teapot_output)
#         dst_d = os.path.join(drive_dest, rel) if rel != '.' else drive_dest
#         os.makedirs(dst_d, exist_ok=True)
#         for f in files:
#             shutil.copy2(os.path.join(root, f), os.path.join(dst_d, f))
#     print(f"✅ Đã đồng bộ thêm bản sao vào: {drive_dest}")
# except Exception as e:
#     print(f"ℹ️ Ghi chú MyDrive path: {e}")

# print(f"\n🎉 Hoàn tất đẩy kết quả '{RUN_FOLDER_NAME}' lên Google Drive!")


📂 Thư mục output RUN 3: /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs/output/anisotropic_synthetic/teapot_only_ref_score
📤 Đang tải 'teapot_only_ref_score' lên Drive (Parent ID: 1FZdVaZmY9VsFMGXWc5107z4pGz-cTyCh)...


✅ Đã upload thành công thư mục 'teapot_only_ref_score' lên Google Drive!
✅ Đã đồng bộ thêm bản sao vào: /content/drive/MyDrive/Thesis/Results/20082026/teapot_only_ref_score

🎉 Hoàn tất đẩy kết quả 'teapot_only_ref_score' lên Google Drive!


## 🏃‍♂️ Bước 11: RUN 4 — Ablation: Dùng --use_ref_score + --use_adaptive_prior

> **Mô tả cấu hình**: Sử dụng Reflection Score kết hợp Adaptive Prior, tắt Reflection View Sampling.
> **Thư mục Output**: `output/anisotropic_synthetic/teapot_ref_score_adaptive`
> **Cơ chế Auto-Skip**: Nếu đã tồn tại `results.json` $\rightarrow$ Bỏ qua trong 0.1s!

In [ ]:
# ── Tạo Script Chạy Cho RUN 4 (exec_run4_teapot_teapot_ref_score_adaptive.sh) ─────────────
import os

run_sh_path = os.path.join(REPO_DIR, "exec_run4_teapot_teapot_ref_score_adaptive.sh")
run_script_content = """#!/bin/bash
set -e
export CUDA_VISIBLE_DEVICES=0

echo "======================================================================="
echo "🚀 BẮT ĐẦU RUN 4: Ablation: Dùng --use_ref_score + --use_adaptive_prior"
echo "======================================================================="

OUTPUT_PATH="output/anisotropic_synthetic/teapot_ref_score_adaptive"

# 1. Kiểm tra Auto-Skip: Nếu đã có results.json thì bỏ qua 100%
if [ -f "$OUTPUT_PATH/results.json" ]; then
    echo "✅ RUN 4 đã hoàn thành đầy đủ trước đó (đã có results.json). Bỏ qua toàn bộ bước này!"
    exit 0
fi

# 2. Trích xuất Reflection Prior (nếu chưa có)
if [ ! -d "datasets/Anisotropic-Synthesis/teapot/reflection_prior" ] || [ $(ls -1 datasets/Anisotropic-Synthesis/teapot/reflection_prior | wc -l) -eq 0 ]; then
    python extract_reflection_prior.py \
        -s datasets/Anisotropic-Synthesis/teapot \
        --eval \
        --white_background \
        --data_device cpu \
        --ref_prior_method tan \
        --ti_thresh 0.35 \
        --ti_bright 0.60
else
    echo "⚡ Reflection Prior đã tồn tại. Bỏ qua bước trích xuất prior!"
fi

# 3. Huấn luyện (Nếu đã có checkpoint 30k thì bỏ qua train.py)
if [ -f "$OUTPUT_PATH/point_cloud/iteration_30000/point_cloud.ply" ]; then
    echo "⚡ Checkpoint 30,000 iterations đã hoàn thành trước đó! Chuyển thẳng sang bước Render..."
else
    python train.py \
        -s datasets/Anisotropic-Synthesis/teapot \
        -m $OUTPUT_PATH \
        --eval \
        --white_background \
        --asg_degree 24 \
        --densification_interval 500 \
        --densification_refscore_interval 500 \
        --num_score_cameras 10 \
        --optimizer_type default \
        --use_ref_score \
        --use_adaptive_prior
        # --use_reflection_view_sampling
        # --disable_multiview_contribution
fi

# 4. Kết xuất ảnh (Render)
python render.py -s datasets/Anisotropic-Synthesis/teapot -m $OUTPUT_PATH --iteration 30000 --skip_train --data_device cpu

# 5. Tính toán Metrics
python metrics.py -m $OUTPUT_PATH

echo "🎉 Hoàn tất RUN 4 (teapot_ref_score_adaptive) thành công!"
"""

with open(run_sh_path, 'w', encoding='utf-8') as f:
    f.write(run_script_content)

!chmod +x "{run_sh_path}"
print(f"📜 Đã khởi tạo script chạy RUN 4: {run_sh_path}")


📜 Đã khởi tạo script chạy RUN 4: /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs/exec_run4_teapot_teapot_ref_score_adaptive.sh


In [ ]:
%%bash
# ── Thực thi RUN 4: Ablation: Dùng --use_ref_score + --use_adaptive_prior ───────────────────────────────────────
set -e
export CUDA_HOME=/usr/local/cuda
export PATH=$CUDA_HOME/bin:$PATH
TORCH_LIB=$(python3 -c 'import torch, os; print(os.path.join(os.path.dirname(torch.__file__), "lib"))' 2>/dev/null || echo "")
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$TORCH_LIB:$LD_LIBRARY_PATH
export CUDA_VISIBLE_DEVICES=0

MYDRIVE_ROOT="/content/drive/MyDrive"
REPO_DIR="$MYDRIVE_ROOT/Thesis/source/20082026/spec-fastgs"
cd "$REPO_DIR"

bash exec_run4_teapot_teapot_ref_score_adaptive.sh


🚀 BẮT ĐẦU RUN 4: Ablation: Dùng --use_ref_score + --use_adaptive_prior
⚡ Reflection Prior đã tồn tại. Bỏ qua bước trích xuất prior!
Output folder already exists and is not empty. Moving old run to: output/anisotropic_synthetic/backups/teapot_ref_score_adaptive/unknown_20260820_161940 [20/08 16:20:25]
Output folder: output/anisotropic_synthetic/teapot_ref_score_adaptive [20/08 16:20:25]
[SH-ASG Ablation] use_asg=True [20/08 16:20:25]
[FastGS Ablation] VCD=True, VCP=True, CompactBox=True, beta=0.5 [20/08 16:20:25]
Detected Blender dataset (transforms_train.json) [20/08 16:20:25]
Reading Training Transforms [20/08 16:20:25]
Reading Test Transforms [20/08 16:20:57]
Loading Training Cameras [20/08 16:21:04]
Loading Test Cameras [20/08 16:21:12]
Number of points at initialisation :  100000 [20/08 16:21:13]
[Auto Budget] Ref Score cap: 1,000,000 Gaussians (initial=100,000, multiplier=10.0) [20/08 16:21:14]
Loading Reflection Priors... [20/08 16:21:14]
Loaded 600 reflection priors; missing 0. 

2026-08-20 16:20:16.435677: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
/content/drive/.shortcut-targets-by-id/1f8jbBEpIwgX61UWWi0ezERkYdRKYGReJ/Thesis/source/20082026/spec-fastgs/train.py:535: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  prior_img = imageio.imread(path)
test Gaussian heatmaps: 100%|██████████| 100/100 [00:11<00:00,  8.66it/s]
fatal: not a git repository (or any parent up to mount

## ☁️ Bước 12: Đẩy Kết Quả RUN 4 Lên Thư Mục Google Drive [Hình 3] (`teapot_ref_score_adaptive`)

> Upload toàn bộ kết quả lên Google Drive Folder ID `1FZdVaZmY9VsFMGXWc5107z4pGz-cTyCh` với tên folder là `teapot_ref_score_adaptive` (**loại trừ hoàn toàn thư mục `test/`**).

In [ ]:
# ── Lưu & Đẩy Kết Quả RUN 4 Lên Drive (Loại trừ folder 'test') ─────────────────
import os
import shutil

RUN_FOLDER_NAME = "teapot_ref_score_adaptive"
run_teapot_output = os.path.join(REPO_DIR, "output", "anisotropic_synthetic", RUN_FOLDER_NAME)
if not os.path.exists(run_teapot_output):
    for root, dirs, files in os.walk(os.path.join(REPO_DIR, "output")):
        if RUN_FOLDER_NAME in root and ("point_cloud" in dirs or "cameras.json" in files or "results.json" in files):
            run_teapot_output = root
            break

print(f"📂 Thư mục output RUN 4: {run_teapot_output}")

# 1. Đẩy trực tiếp qua Google Drive API vào Folder ID ở Hình 3
upload_local_folder_to_drive(
    local_dir=run_teapot_output,
    parent_folder_id=GDRIVE_RESULTS_FOLDER_ID,
    folder_name=RUN_FOLDER_NAME,
    exclude_names=['test']
)

# 2. Đồng thời lưu bản sao vào MyDrive
drive_dest = os.path.join(GDRIVE_RESULTS_DIR, RUN_FOLDER_NAME)
try:
    os.makedirs(drive_dest, exist_ok=True)
    for root, dirs, files in os.walk(run_teapot_output):
        if 'test' in [d.lower() for d in dirs]: dirs.remove('test')
        rel = os.path.relpath(root, run_teapot_output)
        dst_d = os.path.join(drive_dest, rel) if rel != '.' else drive_dest
        os.makedirs(dst_d, exist_ok=True)
        for f in files:
            shutil.copy2(os.path.join(root, f), os.path.join(dst_d, f))
    print(f"✅ Đã đồng bộ thêm bản sao vào: {drive_dest}")
except Exception as e:
    print(f"ℹ️ Ghi chú MyDrive path: {e}")

print(f"\n🎉 Hoàn tất đẩy kết quả '{RUN_FOLDER_NAME}' lên Google Drive!")


📂 Thư mục output RUN 4: /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs/output/anisotropic_synthetic/teapot_ref_score_adaptive
📤 Đang tải 'teapot_ref_score_adaptive' lên Drive (Parent ID: 1FZdVaZmY9VsFMGXWc5107z4pGz-cTyCh)...


✅ Đã upload thành công thư mục 'teapot_ref_score_adaptive' lên Google Drive!
✅ Đã đồng bộ thêm bản sao vào: /content/drive/MyDrive/Thesis/Results/20082026/teapot_ref_score_adaptive

🎉 Hoàn tất đẩy kết quả 'teapot_ref_score_adaptive' lên Google Drive!


## 🏃‍♂️ Bước 13: RUN 5 — Ablation: Tắt Toàn Bộ 3 Cờ Reflection (Vanilla)

> **Mô tả cấu hình**: Huấn luyện không dùng bất kỳ Reflection Prior nào (tắt cả 3 cờ `use_*`).
> **Thư mục Output**: `output/anisotropic_synthetic/teapot_no_reflection`
> **Cơ chế Auto-Skip**: Nếu đã tồn tại `results.json` $\rightarrow$ Bỏ qua trong 0.1s!

In [ ]:
# ── Tạo Script Chạy Cho RUN 5 (exec_run5_teapot_teapot_no_reflection.sh) ─────────────
import os

run_sh_path = os.path.join(REPO_DIR, "exec_run5_teapot_teapot_no_reflection.sh")
run_script_content = """#!/bin/bash
set -e
export CUDA_VISIBLE_DEVICES=0

echo "======================================================================="
echo "🚀 BẮT ĐẦU RUN 5: Ablation: Tắt Toàn Bộ 3 Cờ Reflection (Vanilla)"
echo "======================================================================="

OUTPUT_PATH="output/anisotropic_synthetic/teapot_no_reflection"

# 1. Kiểm tra Auto-Skip: Nếu đã có results.json thì bỏ qua 100%
if [ -f "$OUTPUT_PATH/results.json" ]; then
    echo "✅ RUN 5 đã hoàn thành đầy đủ trước đó (đã có results.json). Bỏ qua toàn bộ bước này!"
    exit 0
fi

# 2. Trích xuất Reflection Prior (nếu chưa có)
if [ ! -d "datasets/Anisotropic-Synthesis/teapot/reflection_prior" ] || [ $(ls -1 datasets/Anisotropic-Synthesis/teapot/reflection_prior | wc -l) -eq 0 ]; then
    python extract_reflection_prior.py \
        -s datasets/Anisotropic-Synthesis/teapot \
        --eval \
        --white_background \
        --data_device cpu \
        --ref_prior_method tan \
        --ti_thresh 0.35 \
        --ti_bright 0.60
else
    echo "⚡ Reflection Prior đã tồn tại. Bỏ qua bước trích xuất prior!"
fi

# 3. Huấn luyện (Nếu đã có checkpoint 30k thì bỏ qua train.py)
if [ -f "$OUTPUT_PATH/point_cloud/iteration_30000/point_cloud.ply" ]; then
    echo "⚡ Checkpoint 30,000 iterations đã hoàn thành trước đó! Chuyển thẳng sang bước Render..."
else
    python train.py \
        -s datasets/Anisotropic-Synthesis/teapot \
        -m $OUTPUT_PATH \
        --eval \
        --white_background \
        --asg_degree 24 \
        --densification_interval 500 \
        --densification_refscore_interval 500 \
        --num_score_cameras 10 \
        --optimizer_type default \

        # --use_ref_score
        # --use_adaptive_prior
        # --use_reflection_view_sampling
        --disable_multiview_contribution
fi

# 4. Kết xuất ảnh (Render)
python render.py -s datasets/Anisotropic-Synthesis/teapot -m $OUTPUT_PATH --iteration 30000 --skip_train --data_device cpu

# 5. Tính toán Metrics
python metrics.py -m $OUTPUT_PATH

echo "🎉 Hoàn tất RUN 5 (teapot_no_reflection) thành công!"
"""

with open(run_sh_path, 'w', encoding='utf-8') as f:
    f.write(run_script_content)

!chmod +x "{run_sh_path}"
print(f"📜 Đã khởi tạo script chạy RUN 5: {run_sh_path}")


📜 Đã khởi tạo script chạy RUN 5: /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs/exec_run5_teapot_teapot_no_reflection.sh


In [ ]:
%%bash
# ── Thực thi RUN 5: Ablation: Tắt Toàn Bộ 3 Cờ Reflection (Vanilla) ───────────────────────────────────────
set -e
export CUDA_HOME=/usr/local/cuda
export PATH=$CUDA_HOME/bin:$PATH
TORCH_LIB=$(python3 -c 'import torch, os; print(os.path.join(os.path.dirname(torch.__file__), "lib"))' 2>/dev/null || echo "")
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$TORCH_LIB:$LD_LIBRARY_PATH
export CUDA_VISIBLE_DEVICES=0

MYDRIVE_ROOT="/content/drive/MyDrive"
REPO_DIR="$MYDRIVE_ROOT/Thesis/source/20082026/spec-fastgs"
cd "$REPO_DIR"

bash exec_run5_teapot_teapot_no_reflection.sh


🚀 BẮT ĐẦU RUN 5: Ablation: Tắt Toàn Bộ 3 Cờ Reflection (Vanilla)
⚡ Reflection Prior đã tồn tại. Bỏ qua bước trích xuất prior!
⚡ Checkpoint 30,000 iterations đã hoàn thành trước đó! Chuyển thẳng sang bước Render...
Looking for config file in output/anisotropic_synthetic/teapot_no_reflection/cfg_args
Config file found: output/anisotropic_synthetic/teapot_no_reflection/cfg_args
Rendering output/anisotropic_synthetic/teapot_no_reflection
Loading trained model at iteration 30000 [20/08 17:23:02]
Detected Blender dataset (transforms_train.json) [20/08 17:23:02]
Reading Training Transforms [20/08 17:23:02]
Reading Test Transforms [20/08 17:23:33]
Loading Test Cameras [20/08 17:23:38]
Loading ASG from: output/anisotropic_synthetic/teapot_no_reflection/point_cloud/iteration_30000/asg.pt [20/08 17:23:40]
ASG loaded with shape: torch.Size([14062, 24]) [20/08 17:23:40]
[test] 100 frames | FPS: 336.45 [20/08 17:24:40]
Saved test FPS to output/anisotropic_synthetic/teapot_no_reflection/results.json 

Metric evaluation progress: 100%|██████████| 100/100 [02:51<00:00,  1.72s/it]


## ☁️ Bước 14: Đẩy Kết Quả RUN 5 Lên Thư Mục Google Drive [Hình 3] (`teapot_no_reflection`)

> Upload toàn bộ kết quả lên Google Drive Folder ID `1FZdVaZmY9VsFMGXWc5107z4pGz-cTyCh` với tên folder là `teapot_no_reflection` (**loại trừ hoàn toàn thư mục `test/`**).

In [ ]:
# ── Lưu & Đẩy Kết Quả RUN 5 Lên Drive (Loại trừ folder 'test') ─────────────────
import os
import shutil

RUN_FOLDER_NAME = "teapot_no_reflection"
run_teapot_output = os.path.join(REPO_DIR, "output", "anisotropic_synthetic", RUN_FOLDER_NAME)
if not os.path.exists(run_teapot_output):
    for root, dirs, files in os.walk(os.path.join(REPO_DIR, "output")):
        if RUN_FOLDER_NAME in root and ("point_cloud" in dirs or "cameras.json" in files or "results.json" in files):
            run_teapot_output = root
            break

print(f"📂 Thư mục output RUN 5: {run_teapot_output}")

# 1. Đẩy trực tiếp qua Google Drive API vào Folder ID ở Hình 3
upload_local_folder_to_drive(
    local_dir=run_teapot_output,
    parent_folder_id=GDRIVE_RESULTS_FOLDER_ID,
    folder_name=RUN_FOLDER_NAME,
    exclude_names=['test']
)

# 2. Đồng thời lưu bản sao vào MyDrive
drive_dest = os.path.join(GDRIVE_RESULTS_DIR, RUN_FOLDER_NAME)
try:
    os.makedirs(drive_dest, exist_ok=True)
    for root, dirs, files in os.walk(run_teapot_output):
        if 'test' in [d.lower() for d in dirs]: dirs.remove('test')
        rel = os.path.relpath(root, run_teapot_output)
        dst_d = os.path.join(drive_dest, rel) if rel != '.' else drive_dest
        os.makedirs(dst_d, exist_ok=True)
        for f in files:
            shutil.copy2(os.path.join(root, f), os.path.join(dst_d, f))
    print(f"✅ Đã đồng bộ thêm bản sao vào: {drive_dest}")
except Exception as e:
    print(f"ℹ️ Ghi chú MyDrive path: {e}")

print(f"\n🎉 Hoàn tất đẩy kết quả '{RUN_FOLDER_NAME}' lên Google Drive!")


📂 Thư mục output RUN 5: /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs/output/anisotropic_synthetic/teapot_no_reflection
📤 Đang tải 'teapot_no_reflection' lên Drive (Parent ID: 1FZdVaZmY9VsFMGXWc5107z4pGz-cTyCh)...
✅ Đã upload thành công thư mục 'teapot_no_reflection' lên Google Drive!
✅ Đã đồng bộ thêm bản sao vào: /content/drive/MyDrive/Thesis/Results/20082026/teapot_no_reflection

🎉 Hoàn tất đẩy kết quả 'teapot_no_reflection' lên Google Drive!


## 📊 Bước Cuối: Bảng Tổng Hợp & So Sánh Kết Quả Toàn Bộ 5 Thực Nghiệm

> Tổng hợp và so sánh toàn diện các chỉ số chất lượng hình ảnh (PSNR, SSIM, LPIPS) và số lượng hạt Gaussians trên scene `teapot` cho cả 5 cấu hình.

In [ ]:
# ── Tổng Hợp & Hiển Thị Bảng So Sánh Số Liệu Cả 5 Thực Nghiệm ──────────────────
import json
import pandas as pd

runs = [
    {"Name": "RUN 1 (Full Baseline)", "Dir": os.path.join(REPO_DIR, "output", "anisotropic_synthetic", "teapot")},
    {"Name": "RUN 2 (Disable Multiview)", "Dir": os.path.join(REPO_DIR, "output", "anisotropic_synthetic", "teapot_disable_multiview")},
    {"Name": "RUN 3 (Only Ref Score)", "Dir": os.path.join(REPO_DIR, "output", "anisotropic_synthetic", "teapot_only_ref_score")},
    {"Name": "RUN 4 (Ref Score + Adaptive)", "Dir": os.path.join(REPO_DIR, "output", "anisotropic_synthetic", "teapot_ref_score_adaptive")},
    {"Name": "RUN 5 (No Reflection / Vanilla)", "Dir": os.path.join(REPO_DIR, "output", "anisotropic_synthetic", "teapot_no_reflection")},
]

summary_rows = []
for r in runs:
    res_file = os.path.join(r["Dir"], "results.json")
    row = {
        "Configuration": r["Name"],
        "Status": "⏳ Chưa chạy",
        "PSNR ↑": "N/A",
        "SSIM ↑": "N/A",
        "LPIPS ↓": "N/A",
        "Gaussians": "N/A"
    }
    if os.path.exists(res_file):
        row["Status"] = "✅ Hoàn thành"
        try:
            with open(res_file, "r") as f:
                data = json.load(f)
            metrics = data.get("ours_30000", {})
            if not metrics and len(data) > 0:
                metrics = list(data.values())[0]

            if "PSNR" in metrics: row["PSNR ↑"] = f"{metrics['PSNR']:.4f}"
            if "SSIM" in metrics: row["SSIM ↑"] = f"{metrics['SSIM']:.4f}"
            if "LPIPS" in metrics: row["LPIPS ↓"] = f"{metrics['LPIPS']:.4f}"

            train_info_file = os.path.join(r["Dir"], "train_info.json")
            if os.path.exists(train_info_file):
                with open(train_info_file, "r") as tf:
                    tinfo = json.load(tf)
                    if "num_gaussians" in tinfo:
                        row["Gaussians"] = f"{tinfo['num_gaussians']:,}"
        except Exception as e:
            print(f"⚠️ Không đọc được {res_file}: {e}")
    elif os.path.exists(os.path.join(r["Dir"], "point_cloud", "iteration_30000", "point_cloud.ply")):
        row["Status"] = "⚡ Đã train (chờ render)"
    summary_rows.append(row)

df = pd.DataFrame(summary_rows)
print("=" * 85)
print("📊 BẢNG TỔNG HỢP KẾT QUẢ 5 THỰC NGHIỆM TRÊN SCENE 'TEAPOT'")
print("=" * 85)
try:
    from IPython.display import display
    display(df)
except Exception:
    print(df.to_string(index=False))

# Lưu file tổng kết Excel & CSV
csv_path = os.path.join(REPO_DIR, "output", "anisotropic_synthetic", "summary_teapot_all_5_runs.csv")
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
df.to_csv(csv_path, index=False)
print(f"\n💾 Đã lưu bảng tổng hợp ra CSV tại: {csv_path}")


📊 BẢNG TỔNG HỢP KẾT QUẢ 5 THỰC NGHIỆM TRÊN SCENE 'TEAPOT'


,Configuration,Status,PSNR ↑,SSIM ↑,LPIPS ↓,Gaussians
0,RUN 1 (Full Baseline),✅ Hoàn thành,33.3377,0.9829,0.0280,N/A
1,RUN 2 (Disable Multiview),✅ Hoàn thành,35.3981,0.9871,0.0218,N/A
2,RUN 3 (Only Ref Score),✅ Hoàn thành,32.7146,0.9816,0.0292,N/A
3,RUN 4 (Ref Score + Adaptive),✅ Hoàn thành,33.4234,0.9830,0.0277,N/A
4,RUN 5 (No Reflection / Vanilla),✅ Hoàn thành,32.6882,0.9815,0.0293,N/A



💾 Đã lưu bảng tổng hợp ra CSV tại: /content/drive/MyDrive/Thesis/source/20082026/spec-fastgs/output/anisotropic_synthetic/summary_teapot_all_5_runs.csv
